In [63]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, OrdinalEncoder, OneHotEncoder
)
from sklearn.impute import SimpleImputer


In [64]:
df = pd.read_csv("../data/raw/Loan_Default.csv")

In [65]:
df.drop("ID", axis=1, inplace=True)

In [66]:
df.head()

,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,116500,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,206500,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,406500,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,456500,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,696500,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [67]:
df["Security_Type"] = df["Security_Type"].replace("Indriect", "Indirect")

In [68]:
mnar_columns = ["rate_of_interest", "Interest_rate_spread", "Upfront_charges", "dtir1"]

for col in mnar_columns:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(df[col].median())
    

In [69]:
df.shape

(148670, 37)

In [70]:
df.head()

,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,...,submission_of_application,LTV,Region,Security_Type,Status,dtir1,rate_of_interest_missing,Interest_rate_spread_missing,Upfront_charges_missing,dtir1_missing
0,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,116500,...,to_inst,98.728814,south,direct,1,45.0,1,1,1,0
1,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,206500,...,to_inst,NaN,North,direct,1,39.0,1,1,1,1
2,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,406500,...,to_inst,80.019685,south,direct,0,46.0,0,0,0,0
3,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,456500,...,not_inst,69.376900,North,direct,0,42.0,0,0,1,0
4,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,696500,...,not_inst,91.886544,North,direct,0,39.0,0,0,0,0


In [71]:
df["property_value"] = df["property_value"].fillna(df["property_value"].median())

df["LTV"] = df["loan_amount"] / df["property_value"]

# Encode features

In [72]:
ordinal_cols = ["age", "total_units"]
ordinal_categories = [
    ["<25", "25-34", "35-44", "45-54", "55-64", "65-74", ">74"],
    ["1U", "2U", "3U", "4U"]
]


binary_cols = ["loan_limit", "approv_in_adv", "Credit_Worthiness", "open_credit", "business_or_commercial", "Neg_ammortization", "interest_only", "lump_sum_payment", "construction_type", "Secured_by", "co-applicant_credit_type", "submission_of_application", "Security_Type"]

nominal_cols = ["Gender", "loan_type", "loan_purpose", "occupancy_type", "credit_type", "Region"]

numeric_cols = ["year", "loan_amount", "rate_of_interest", "Interest_rate_spread", "Upfront_charges", "term", "property_value", "income", "Credit_Score", "LTV", "dtir1"]

passthrough_cols = ["rate_of_interest_missing", "Interest_rate_spread_missing", "Upfront_charges_missing", "dtir1_missing"]

In [73]:
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [74]:
onehotencode_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore",
            drop='first'
            ))
    ])

In [75]:
ordinal_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories=ordinal_categories,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    ))
])

In [76]:
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        categories='auto'
    ))
])

In [77]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_pipeline, numeric_cols),
        ("ord", ordinal_pipeline, ordinal_cols),
        ("bin", binary_pipeline, binary_cols),
        ("nom", onehotencode_pipeline, nominal_cols),
        ("pass", "passthrough", passthrough_cols)
    ]
)

In [78]:
X = df.drop('Status', axis=1)
y = df["Status"]

In [79]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y , test_size=0.2, random_state=42, stratify=y
)

In [80]:
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

In [81]:
X_train_transformed.shape

(118936, 46)

array([[ 0.        , -0.67965894, -2.36445142, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        ,  0.73982819,  0.44681625, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        , -0.02451103, -0.08476891, ...,  1.        ,
         1.        ,  0.        ],
       ...,
       [ 0.        ,  1.99552835, -0.83103269, ...,  0.        ,
         0.        ,  0.        ],
       [ 0.        , -0.51587197, -0.08476891, ...,  1.        ,
         1.        ,  0.        ],
       [ 0.        ,  2.05012401, -0.31989311, ...,  0.        ,
         0.        ,  1.        ]], shape=(118936, 46))

In [85]:
# See what OneHotEncoder actually created
ohe = preprocessor.named_transformers_['nom']['encoder']
print(ohe.get_feature_names_out(nominal_cols))


['Gender_Joint' 'Gender_Male' 'Gender_Sex Not Available' 'loan_type_type2'
 'loan_type_type3' 'loan_purpose_p2' 'loan_purpose_p3' 'loan_purpose_p4'
 'occupancy_type_pr' 'occupancy_type_sr' 'credit_type_CRIF'
 'credit_type_EQUI' 'credit_type_EXP' 'Region_North-East' 'Region_central'
 'Region_south']


In [86]:
df.groupby('Gender')['Status'].mean()

Gender
Female               0.251155
Joint                0.191623
Male                 0.261914
Sex Not Available    0.285908
Name: Status, dtype: float64